In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

import seaborn as sns

import json
import pathlib as pl

import numpy as np
import pandas as pd
import anndata as ad
import geopandas as gpd
from shapely.geometry import Point

import matplotlib.pyplot as plt
import math

In [ ]:
import matplotlib
matplotlib.rcParams['svg.fonttype'] = 'none'

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

import torch
print(torch.cuda.get_device_name(0))

# Download data

In [ ]:
def build_palettes_from_adata(adata, palette_specs):
    """
    Build labeled color palettes for categorical columns in adata.obs.

    Parameters
    ----------
    adata : AnnData
        Must have .obs DataFrame containing categorical columns.
    palette_specs : dict
        Mapping {column_name: palette} where palette can be:
          - a string palette name (e.g. "tab10")
          - a list of RGB colors (custom)

    Returns
    -------
    dict
        {column_name: {label: color}} mapping.
    """
    custom_palettes = {}

    for col, palette in palette_specs.items():
        if col not in adata.obs.columns:
            print(f"⚠️ Warning: '{col}' not found in adata.obs — skipping.")
            continue

        unique_vals = sorted(adata.obs[col].astype(str).dropna().unique())
        n_unique = len(unique_vals)

        # If user passed a name → generate via seaborn
        if isinstance(palette, str):
            pal_colors = sns.color_palette(palette, n_colors=n_unique)
        # If user passed a list → use directly
        elif isinstance(palette, (list, tuple)):
            pal_colors = palette[:n_unique]
        else:
            raise ValueError(f"Unsupported palette type for '{col}': {type(palette)}")

        color_dict = dict(zip(unique_vals, pal_colors))
        custom_palettes[col] = color_dict

    print(f"✅ Built palettes for {len(custom_palettes)} columns.")
    return custom_palettes


def plot_celltype_spatial_single_split_legend(
    df,
    color_by="celltype",
    sample_id=None,
    title=None,
    palette_dict=None,         # ✅ added
    palette_name="tab20",
    s=1.5,
    save_svg=True,
    output_prefix="spatial_plot",
    legend_title=None,
):
    """
    Nature Genetics–style spatial scatterplot for one sample,
    saving main plot as PNG (raster) and legend separately as SVG (vector).
    """
    sns.set_style("white")
    sns.set_context("talk")

    # --- Subset one sample ---
    if sample_id is not None:
        df = df[df["sample_id"] == sample_id].copy()
        if df.empty:
            raise ValueError(f"Sample ID '{sample_id}' not found in DataFrame.")

    # --- Colors ---
    unique_labels = sorted(df[color_by].dropna().unique())
    if palette_dict is not None and color_by in palette_dict:
        color_dict = palette_dict[color_by]
    else:
        palette = sns.color_palette(palette_name, n_colors=len(unique_labels))
        color_dict = dict(zip(unique_labels, palette))

    # --- Main plot ---
    fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
    sns.scatterplot(
        data=df,
        x="X_coord", y="Y_coord",
        hue=color_by, palette=color_dict,
        s=s, alpha=0.9, linewidth=0,
        rasterized=True, ax=ax, legend=False
    )
    ax.invert_yaxis(); ax.set_aspect("equal", adjustable="box")
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(""); ax.set_ylabel("")
    plt.tight_layout()

    # --- Save main figure ---
    fname_main = f"{output_prefix}_{sample_id or 'sample'}_main.png"
    fig.savefig(fname_main, dpi=300, bbox_inches="tight", transparent=True, format="png")
    print(f"Saved main figure: {fname_main}")

    # --- Legend ---
    fig_leg, ax_leg = plt.subplots(figsize=(3, 0.5 * len(unique_labels)), dpi=300)
    handles = [
        plt.Line2D([0], [0], marker='o', color='none', label=label,
                   markerfacecolor=color_dict[label], markersize=8)
        for label in unique_labels
    ]
    ax_leg.legend(handles=handles, loc="center left", frameon=False,
                  title=legend_title or color_by, title_fontsize=14, fontsize=14)
    ax_leg.axis("off")
    plt.tight_layout()

    if save_svg:
        fname_leg = f"{output_prefix}_{sample_id or 'sample'}_legend.svg"
        fig_leg.savefig(fname_leg, dpi=300, bbox_inches="tight", transparent=True, format="svg")
        print(f"Saved legend: {fname_leg}")

    plt.close(fig); plt.close(fig_leg)


In [ ]:
rawdata = sc.read_h5ad('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Ovarian_5k/adata.h5ad')

region_annot = pd.read_csv('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Ovarian_5k/region_annotations.csv',index_col=0)

rawdata.obs['path_region'] = region_annot.loc[rawdata.obs_names].values.ravel()

rawdata.obs = pd.concat([rawdata.obs, pd.DataFrame(rawdata.obsm['spatial_px'], index=rawdata.obs_names, columns=['X_coord','Y_coord'])],axis=1)

region_df = rawdata.obs[['cell_labels', 'minor_celltype', 'major_celltype', 'cell_id',
       'path_region', 'X_coord','Y_coord']]

region_df['sample_id'] = 'TENXOv5k'

In [ ]:
tab_filtered = sns.color_palette()
tab_filtered = [c for i,c in enumerate(tab_filtered) if i not in [4,6]]

tab20_filtered = sns.color_palette('tab20') + sns.color_palette('tab20c')[:11]
tab20_filtered = [c for i,c in enumerate(tab20_filtered) if i not in [8,9,12,13]]

In [ ]:
palette_specs = {
            "path_region": tab_filtered,
            "major_celltype": tab_filtered,
            "minor_celltype": tab20_filtered,
        }

palette_dict_1 = build_palettes_from_adata(rawdata, palette_specs)

In [ ]:
cpal = sns.color_palette()
palette_dict_1['major_celltype'] = {'Malignant': cpal[0], 'Epithelial': cpal[1], 'Myeloid': cpal[2],
                  'Lymphocytes': cpal[3], 'Stromal': cpal[5], 'Endothelial': cpal[9],
                  'Pericytes': cpal[8], 'Unassigned': cpal[7]}

# Get embeddings

In [ ]:
from spatialfusion.embed.embed import AEInputs, run_full_embedding

In [ ]:
basepath = pl.Path('../../../Broad_SpatialFoundation/test_data/')
sample_name = '10X_Xenium_Ovarian_5k'
output_dir = basepath / sample_name

adata = sc.read_h5ad(basepath / sample_name / 'adata.h5ad')
adata.obs = pd.concat([adata.obs, pd.DataFrame(adata.obsm['spatial_px'], index=adata.obs_names, columns=['X_coord','Y_coord'])],axis=1)
adata.obs["sample_id"] = sample_name

pathway_matrix = pd.read_parquet(basepath / sample_name / 'pathway_activation.parquet')

pathways_by_sample = {
    sample_name: pathway_matrix,
}

In [ ]:
uni_df = pd.read_parquet(pl.Path(output_dir) / 'embeddings' / 'UNI.parquet')
scgpt_df = pd.read_parquet(pl.Path(output_dir) / 'embeddings' / 'scGPT.parquet')

In [ ]:
ae_inputs_by_sample = {
    sample_name: AEInputs(adata=adata, z_he=uni_df, z_rna=scgpt_df),
}

In [ ]:
outdir = pl.Path("../../../SpatialFusion/results/gcn_pathway_ablation_sweep/GCN_embeddings/")
os.makedirs(outdir, exist_ok=True)

# Minus Androgen

In [ ]:
# this uses the average version
embeddings_df = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    ae_model_path='../../../SpatialFusion/results/ae_encoder_sweep/uni_scgpt_full_20260602-024030_1fbc1fe4/model.pt',
    gcn_model_path='../../../SpatialFusion/results/gcn_pathway_ablation_sweep/uni_scgpt_full_gcn_avg_reg_cls_ablation_Androgen_8fb6298f/model.pt',
    device="cuda:0",
    combine_mode="average",
    spatial_key='spatial_px',
    celltype_key='major_celltype',
    save_ae_dir=None,  # optional
)

In [ ]:
out_path = outdir / "minus_Androgen.parquet"
embeddings_df.to_parquet(out_path)

# Minus EGFR

In [ ]:
# this uses the average version
embeddings_df = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    ae_model_path='../../../SpatialFusion/results/ae_encoder_sweep/uni_scgpt_full_20260602-024030_1fbc1fe4/model.pt',
    gcn_model_path='../../../SpatialFusion/results/gcn_pathway_ablation_sweep/uni_scgpt_full_gcn_avg_reg_cls_ablation_EGFR_a8254962/model.pt',
    device="cuda:0",
    combine_mode="average",
    spatial_key='spatial_px',
    celltype_key='major_celltype',
    save_ae_dir=None,  # optional
)

In [ ]:
out_path = outdir / "minus_EGFR.parquet"
embeddings_df.to_parquet(out_path)

# Minus Estrogen

In [ ]:
# this uses the average version
embeddings_df = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    ae_model_path='../../../SpatialFusion/results/ae_encoder_sweep/uni_scgpt_full_20260602-024030_1fbc1fe4/model.pt',
    gcn_model_path='../../../SpatialFusion/results/gcn_pathway_ablation_sweep/uni_scgpt_full_gcn_avg_reg_cls_ablation_Estrogen_49c41484/model.pt',
    device="cuda:0",
    combine_mode="average",
    spatial_key='spatial_px',
    celltype_key='major_celltype',
    save_ae_dir=None,  # optional
)

In [ ]:
out_path = outdir / "minus_Estrogen.parquet"
embeddings_df.to_parquet(out_path)

# Minus JAK-STAT

In [ ]:
# this uses the average version
embeddings_df = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    ae_model_path='../../../SpatialFusion/results/ae_encoder_sweep/uni_scgpt_full_20260602-024030_1fbc1fe4/model.pt',
    gcn_model_path='../../../SpatialFusion/results/gcn_pathway_ablation_sweep/uni_scgpt_full_gcn_avg_reg_cls_ablation_JAK-STAT_f20658f7/model.pt',
    device="cuda:0",
    combine_mode="average",
    spatial_key='spatial_px',
    celltype_key='major_celltype',
    save_ae_dir=None,  # optional
)

In [ ]:
out_path = outdir / "minus_JAK-STAT.parquet"
embeddings_df.to_parquet(out_path)

# Minus NFkB

In [ ]:
# this uses the average version
embeddings_df = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    ae_model_path='../../../SpatialFusion/results/ae_encoder_sweep/uni_scgpt_full_20260602-024030_1fbc1fe4/model.pt',
    gcn_model_path='../../../SpatialFusion/results/gcn_pathway_ablation_sweep/uni_scgpt_full_gcn_avg_reg_cls_ablation_NFkB_60241b92/model.pt',
    device="cuda:0",
    combine_mode="average",
    spatial_key='spatial_px',
    celltype_key='major_celltype',
    save_ae_dir=None,  # optional
)

In [ ]:
out_path = outdir / "minus_NFkB.parquet"
embeddings_df.to_parquet(out_path)

# Minus PI3K

In [ ]:
# this uses the average version
embeddings_df = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    ae_model_path='../../../SpatialFusion/results/ae_encoder_sweep/uni_scgpt_full_20260602-024030_1fbc1fe4/model.pt',
    gcn_model_path='../../../SpatialFusion/results/gcn_pathway_ablation_sweep/uni_scgpt_full_gcn_avg_reg_cls_ablation_PI3K_04b94156/model.pt',
    device="cuda:0",
    combine_mode="average",
    spatial_key='spatial_px',
    celltype_key='major_celltype',
    save_ae_dir=None,  # optional
)

In [ ]:
out_path = outdir / "minus_PI3K.parquet"
embeddings_df.to_parquet(out_path)

# Minus TGFb

In [ ]:
# this uses the average version
embeddings_df = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    ae_model_path='../../../SpatialFusion/results/ae_encoder_sweep/uni_scgpt_full_20260602-024030_1fbc1fe4/model.pt',
    gcn_model_path='../../../SpatialFusion/results/gcn_pathway_ablation_sweep/uni_scgpt_full_gcn_avg_reg_cls_ablation_TGFb_e2095f7f/model.pt',
    device="cuda:0",
    combine_mode="average",
    spatial_key='spatial_px',
    celltype_key='major_celltype',
    save_ae_dir=None,  # optional
)

In [ ]:
out_path = outdir / "minus_TGFb.parquet"
embeddings_df.to_parquet(out_path)

# Minus TNFa

In [ ]:
# this uses the average version
embeddings_df = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    ae_model_path='../../../SpatialFusion/results/ae_encoder_sweep/uni_scgpt_full_20260602-024030_1fbc1fe4/model.pt',
    gcn_model_path='../../../SpatialFusion/results/gcn_pathway_ablation_sweep/uni_scgpt_full_gcn_avg_reg_cls_ablation_TNFa_22c674fd/model.pt',
    device="cuda:0",
    combine_mode="average",
    spatial_key='spatial_px',
    celltype_key='major_celltype',
    save_ae_dir=None,  # optional
)

In [ ]:
out_path = outdir / "minus_TNFa.parquet"
embeddings_df.to_parquet(out_path)

# Minus VEGF

In [ ]:
# this uses the average version
embeddings_df = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    ae_model_path='../../../SpatialFusion/results/ae_encoder_sweep/uni_scgpt_full_20260602-024030_1fbc1fe4/model.pt',
    gcn_model_path='../../../SpatialFusion/results/gcn_pathway_ablation_sweep/uni_scgpt_full_gcn_avg_reg_cls_ablation_VEGF_255149c5/model.pt',
    device="cuda:0",
    combine_mode="average",
    spatial_key='spatial_px',
    celltype_key='major_celltype',
    save_ae_dir=None,  # optional
)

In [ ]:
out_path = outdir / "minus_VEGF.parquet"
embeddings_df.to_parquet(out_path)

# Now compare results

# First cluster

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

import scanpy as sc

from scipy.spatial import distance
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import *


In [ ]:
def compute_PAS_fast(clusterlabel, location, k=10):
    clusterlabel = np.array(clusterlabel)
    location = np.array(location)

    # Fit NearestNeighbors (ignore self-match later)
    nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto').fit(location)
    distances, indices = nbrs.kneighbors(location)

    # Remove self (first column is self in most cases)
    neighbor_indices = indices[:, 1:]  # shape: (n_samples, k)

    # Check PAS condition
    mismatches = np.array([
        np.sum(clusterlabel[neighbor_indices[i]] != clusterlabel[i]) > (k / 2)
        for i in range(len(clusterlabel))
    ])

    return np.sum(mismatches) / len(clusterlabel)


def compute_CHAOS_fast(clusterlabel, location):
    clusterlabel = np.array(clusterlabel)
    location = np.array(location)
    matched_location = StandardScaler().fit_transform(location)

    clusterlabel_unique = np.unique(clusterlabel)
    dist_val = 0
    total_count = 0

    for k in tqdm(clusterlabel_unique, desc="Computing CHAOS"):
        cluster_mask = clusterlabel == k
        location_cluster = matched_location[cluster_mask]
        n = location_cluster.shape[0]

        if n <= 2:
            continue

        # Use NearestNeighbors to find 1-NN distances
        nbrs = NearestNeighbors(n_neighbors=2, algorithm='auto').fit(location_cluster)
        distances, _ = nbrs.kneighbors(location_cluster)

        # distances[:, 0] is zero (self), distances[:, 1] is nearest neighbor
        dist_val += np.sum(distances[:, 1])
        total_count += n

    return dist_val / total_count if total_count > 0 else np.nan


def compute_ASW_fast(adata, pred_key, spatial_key='spatial'):
    coords = adata.obsm[spatial_key]
    labels = adata.obs[pred_key]
    return silhouette_score(X=coords, labels=labels, metric='euclidean')

def compute_ARI(adata,gt_key,pred_key):
        return adjusted_rand_score(adata.obs[gt_key],adata.obs[pred_key])

def compute_NMI(adata,gt_key,pred_key):
    return normalized_mutual_info_score(adata.obs[gt_key],adata.obs[pred_key])

def compute_HOM(adata,gt_key,pred_key):
    return homogeneity_score(adata.obs[gt_key],adata.obs[pred_key])

def compute_COM(adata,gt_key,pred_key):
    return completeness_score(adata.obs[gt_key],adata.obs[pred_key])

In [ ]:
adata = rawdata.copy()

In [ ]:
outdir = pl.Path("../../../SpatialFusion/results/gcn_pathway_ablation_sweep/GCN_embeddings/")

In [ ]:
list_pathways = ["Androgen","EGFR","Estrogen","JAK-STAT","NFkB","PI3K","TGFb","TNFa","VEGF"]

In [ ]:
embs_df = {}

for path in list_pathways:
    embs_df[path] = pd.read_parquet(outdir / f"minus_{path}.parquet")

In [ ]:
for path in list_pathways:
    adata.obsm[path] = embs_df[path].set_index('cell_id').loc[adata.obs_names,['0','1','2','3','4','5','6','7','8','9']]

In [ ]:
sc.pp.neighbors(adata, use_rep = "Androgen")

In [ ]:
sc.tl.leiden(adata, resolution=0.14, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs["Androgen"] = adata.obs.leiden.replace({'13': '12', '14': '12', '15': '12', '16': '12', '17': '12', '18': '12', })

In [ ]:
sc.pp.neighbors(adata, use_rep = "EGFR")

In [ ]:
sc.tl.leiden(adata, resolution=0.15, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['EGFR'] = adata.obs.leiden.replace({'13': '12', '14': '12', '15': '12', '16': '12', })

In [ ]:
sc.pp.neighbors(adata, use_rep = "Estrogen")

In [ ]:
sc.tl.leiden(adata, resolution=0.13, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['Estrogen'] = adata.obs.leiden.replace({'13': '12', '14': '12', '15': '12',})

In [ ]:
sc.pp.neighbors(adata, use_rep = "JAK-STAT")

In [ ]:
sc.tl.leiden(adata, resolution=0.13, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['JAK-STAT'] = adata.obs.leiden.replace({'13': '12', '14': '12', '15': '12', '16': '12',  })

In [ ]:
sc.pp.neighbors(adata, use_rep = "NFkB")

In [ ]:
sc.tl.leiden(adata, resolution=0.16, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['NFkB'] = adata.obs.leiden.replace({'13': '12', '14': '12', '15': '12', '16': '12', '17': '12', '18': '12',})

In [ ]:
sc.pp.neighbors(adata, use_rep = "PI3K")

In [ ]:
sc.tl.leiden(adata, resolution=0.15, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['PI3K'] = adata.obs.leiden.replace({'13': '12', '14': '12', '15': '12', '16': '12', '17': '12', '18': '12', })

In [ ]:
sc.pp.neighbors(adata, use_rep = "TGFb")

In [ ]:
sc.tl.leiden(adata, resolution=0.12, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['TGFb'] = adata.obs.leiden.replace({'13': '12', '14': '12', '15': '12', '16': '12', '17': '12', '18': '12',})

In [ ]:
sc.pp.neighbors(adata, use_rep = "TNFa")

In [ ]:
sc.tl.leiden(adata, resolution=0.13, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['TNFa'] = adata.obs.leiden.replace({'13': '12', '14': '12', '15': '12', '16': '12', '17': '12', '18': '12', '19': '12',})

In [ ]:
sc.pp.neighbors(adata, use_rep = "VEGF")

In [ ]:
sc.tl.leiden(adata, resolution=0.135, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['VEGF'] = adata.obs.leiden.replace({'13': '12', '14': '12', '15': '12', '16': '12',
                                              '17': '12', '18': '12', '19': '12', '20': '12',})

# ADD THE ORIGINAL CLUSTERS

In [ ]:
adata_obs_orig= pd.read_csv('../../../SpatialFusion/spatialfusion-analysis/Fig2/benchmark_ovarian_adata_obs_afterreview.csv',index_col=0)

In [ ]:
adata.obs['Full'] = adata_obs_orig.loc[adata.obs_names, 'leiden_gcn']

In [ ]:
#adata.obs.to_csv('benchmark_ovarian_pathway_ablation_ORIGENCODER.csv')
adata.obs.to_csv('benchmark_ovarian_pathway_ablation.csv')

# compute metrics

In [ ]:
def compute_all_metrics(adata, clustering_keys, ground_truth_key='path_region', spatial_key='spatial_px'):
    results = {}

    for method_name, cluster_key in clustering_keys.items():
        metrics = {
            'ARI': compute_ARI(adata, cluster_key, ground_truth_key),
            'NMI': compute_NMI(adata, cluster_key, ground_truth_key),
            'HOM': compute_HOM(adata, cluster_key, ground_truth_key),
            'COM': compute_COM(adata, cluster_key, ground_truth_key),
            'PAS': compute_PAS_fast(adata.obs[cluster_key], adata.obsm[spatial_key]),
            'CHAOS': compute_CHAOS_fast(adata.obs[cluster_key], adata.obsm[spatial_key]),
        }
        results[method_name] = metrics

    return pd.DataFrame(results)

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

def format_number(value):
    """Format numbers: scientific notation if <0.01, else 2 decimals."""
    if pd.isna(value):
        return ""
    if abs(value) < 0.01 and value != 0:
        return f"{value:.0e}"  # 1 decimal in scientific notation, e.g. 3.4e-04
    else:
        return f"{value:.2f}"  # two decimals otherwise

def plot_benchmark_heatmap(
    results_df,
    title="Spatial clustering benchmark",
    savefig=None,
    metric_order=None,
):
    """
    Nature Genetics–style benchmarking heatmap showing method rankings across metrics.
    Allows manual control of metric order.
    """

    lower_better = {'PAS', 'CHAOS'}

    # --- Default metric order ---
    if metric_order is None:
        metric_order = list(results_df.index)

    # --- Normalize scores ---
    df_norm = results_df.copy()
    for metric in df_norm.index:
        vals = df_norm.loc[metric]
        if metric in lower_better:
            vals = -vals
        df_norm.loc[metric] = (vals - vals.min()) / (vals.max() - vals.min() + 1e-9)

    # --- Rank per metric ---
    ranks = results_df.copy()
    for metric in ranks.index:
        ranks.loc[metric] = results_df.loc[metric].rank(ascending=(metric in lower_better))

    # --- Prepare longform for plotting ---
    df_plot = df_norm.reset_index().melt(
        id_vars='index', var_name='Method', value_name='Normalized'
    ).rename(columns={'index': 'Metric'})

    df_plot['Raw'] = results_df.reset_index().melt(
        id_vars='index', var_name='Method', value_name='Raw'
    )['Raw']

    df_plot['Rank'] = ranks.reset_index().melt(
        id_vars='index', var_name='Method', value_name='Rank'
    )['Rank']

    # Add directional arrows
    df_plot['MetricLabel'] = df_plot['Metric'].apply(
        lambda m: f"{m} {'↓' if m in lower_better else '↑'}"
    )

    # --- Construct ordered MetricLabel list ---
    metric_order_labels = []
    for m in metric_order:
        arrow = '↓' if m in lower_better else '↑'
        metric_order_labels.append(f"{m} {arrow}")

    # --- Heatmap data matrix ---
    method_order = results_df.columns.tolist()
    df_matrix = df_plot.pivot_table(
        index="MetricLabel", columns="Method", values="Normalized"
    ).loc[metric_order_labels, method_order]

    # --- Aesthetics ---
    sns.set_theme(style="white", context="talk")

    fig, ax = plt.subplots(figsize=(1.3 * len(method_order), 0.8 * len(metric_order)), dpi=300)
    # Enhance contrast near the top (gamma correction)
    gamma = 3  ### THIS IS ONLY FOR THE COLOR FOR PLOTTING PURPOSES, NOT THE NUMBERS!
    df_matrix_contrast = df_matrix ** gamma
    sns.heatmap(
        df_matrix_contrast,
        #cmap="vlag",
        cmap = LinearSegmentedColormap.from_list(
            "vlag_red",
            ["#fee8ef",  # very light pink
             "#f4a3a8",  # pastel red
             "#d95858",  # mid red
             "#b40426"]  # vlag red (vivid crimson)
        ),
        cbar=False,
        ax=ax,
        linewidths=0,
        square=True,
    )

    # --- Adaptive text color (white on dark, black on light) ---
    #cmap = plt.get_cmap("vlag")
    cmap = LinearSegmentedColormap.from_list(
        "vlag_red",
        ["#fee8ef",  # very light pink
         "#f4a3a8",  # pastel red
         "#d95858",  # mid red
         "#b40426"]  # vlag red (vivid crimson)
    )

    for i, metric in enumerate(df_matrix.index):
        base_metric = metric.split()[0]
        for j, method in enumerate(df_matrix.columns):
            raw_val = results_df.loc[base_metric, method]
            norm_val = df_matrix.loc[metric, method]

            # Compute luminance for adaptive color
            rgb = np.array(cmap(norm_val)[:3])
            luminance = 0.2126 * rgb[0] + 0.7152 * rgb[1] + 0.0722 * rgb[2]
            text_color = "black" if luminance > 0.5 else "white"

            ax.text(
                j + 0.5, i + 0.5,
                format_number(raw_val),
                ha='center', va='center',
                color=text_color,
                fontsize=8,
                fontweight='normal',
            )

    # --- Formatting ---
    ax.set_title(title, fontsize=10, pad=14, fontweight='normal')
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=10, fontweight='normal')
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=10, fontweight='normal')

    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.tight_layout()

    if savefig:
        fig.savefig(
            savefig,
            bbox_inches="tight",
            dpi=300,
            format=savefig.split('.')[-1],
            transparent=True
        )
        print(f"Saved: {savefig}")

    plt.show()



In [ ]:
clustering_keys = {
    f" - {path}": path for path in list_pathways
}
clustering_keys['All pathways']= 'Full'

results_df = compute_all_metrics(adata, clustering_keys)


In [ ]:
higher_better = ["ARI", "NMI", "HOM", "COM"]
lower_better = ["PAS", "CHAOS"]

all_col = "All pathways"
pathways = [c for c in results_df.columns if c != all_col]

impact = pd.Series(0.0, index=pathways)

for metric in higher_better:
    ref = results_df.loc[metric, all_col]
    impact += (ref - results_df.loc[metric, pathways]) / ref

for metric in lower_better:
    ref = results_df.loc[metric, all_col]
    impact += (results_df.loc[metric, pathways] - ref) / ref

pathway_order = impact.sort_values(ascending=False).index

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --------------------------------------------------
# Nature-style plotting parameters
# --------------------------------------------------

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 10,
    "axes.titleweight": "bold",
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "svg.fonttype": "none",   # editable text in Illustrator
})

sns.set_style("white")

# --------------------------------------------------
# Inputs
# --------------------------------------------------

metrics = ["ARI", "NMI", "HOM", "COM", "PAS", "CHAOS"]

higher_better = ["ARI", "NMI", "HOM", "COM"]
lower_better = ["PAS", "CHAOS"]

all_col = "All pathways"

# --------------------------------------------------
# Compute overall pathway importance
# (average normalized performance drop)
# --------------------------------------------------

pathways = [c for c in results_df.columns if c != all_col]

impact = pd.Series(0.0, index=pathways)

for metric in higher_better:
    ref = results_df.loc[metric, all_col]
    impact += (ref - results_df.loc[metric, pathways]) / ref

for metric in lower_better:
    ref = results_df.loc[metric, all_col]
    impact += (results_df.loc[metric, pathways] - ref) / ref

pathway_order = impact.sort_values(ascending=False).index

# --------------------------------------------------
# Plot
# --------------------------------------------------

fig, axes = plt.subplots(
    2,
    3,
    figsize=(7.2, 5.5),   # compact journal-style figure
    constrained_layout=True
)

axes = axes.flatten()

for i, (ax, metric) in enumerate(zip(axes, metrics)):

    vals = results_df.loc[metric, pathway_order]
    ref = results_df.loc[metric, all_col]

    # Highlight most important pathway
    colors = ["#d0d0d0"] * len(vals)

    bars = ax.barh(
        pathway_order,
        vals.values,
        color=colors,
        edgecolor="none"
    )

    # Reference performance
    ax.axvline(
        ref,
        color="red",
        linestyle="--",
        linewidth=1.2,
        zorder=10,
        label="All pathways" if i == 0 else None
    )

    # Percent drop annotations
    xspan = ax.get_xlim()[1] - ax.get_xlim()[0]

    for bar, v in zip(bars, vals):

        if metric in higher_better:
            pct = 100 * (ref - v) / ref
        else:
            pct = 100 * (v - ref) / ref

        ax.text(
            v + xspan * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{pct:.0f}%",
            va="center",
            ha="left",
            fontsize=6.5
        )

    # Titles
    ax.set_title(metric, pad=4)

    # Clean axes
    ax.set_xlabel("")
    ax.set_ylabel("")

    # Only show pathway labels on left column
    if i not in [0, 3]:
        ax.set_yticklabels([])

    sns.despine(ax=ax)

# Legend
axes[0].legend(
    frameon=False,
    loc="upper left",
    handlelength=1.5
)

# Save
plt.savefig(
    "pathway_ablation.svg",
    bbox_inches="tight",
    dpi=300
)

plt.show()